# **Durability emulator — dataset generation**

This notebook **only** runs the emulator and saves the resulting lambda datasets — no PCE is fitted
here. [`02_train_pce.ipynb`](02_train_pce.ipynb) only reads what this notebook writes.

The simulator is the closed-form carbonation model of Possan et al. (2016),
`carbonation_depth_possan_by_type` — no trained ML model is loaded.

**Artefacts written per time step**,
`<n_latent_samples>_<kind>_<split>_<t>_install_<year>_cement_<type>_exposure_<exposure>.pkl`:

| kind | split | content |
|---|---|---|
| `dataset_full` | train / val | one row per latent replica: inputs, effective values, carbonation depth, $g$, lambdas, processing time |
| `dataset_unique` | train / val | one row per design point: inputs, the four lambdas, processing time |

Plus one aggregate `<n_latent_samples>_emulator_timing_durability.pkl`, read back by
[`02_train_pce.ipynb`](02_train_pce.ipynb) to compute the emulator/surrogate speed-up.

CO₂ uses the published CMIP6/SSP table. Set `installation_year` and `co2_scenario` below.
The 100-year horizon must end no later than 2100. Regenerate datasets and retrain after changing the scenario; old polynomial results are not compatible.


## **1. Libraries**

In [8]:
import sys
import time
from pathlib import Path

# functions.py sits one directory up
sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *
from UQpy.distributions import Uniform, JointIndependent

## **2. Random variables and fixed parameters**

Design variables (compressive strength, relative humidity, cover) plus everything the emulator
needs that isn't a design variable.

In [9]:
fck_min = 20  # MPa
fck_max = 50  # MPa
rh_min  = 20  # %
rh_max  = 80  # %
cov_min = 15  # mm
cov_max = 60  # mm

cement_type          = 3
installation_year    = 1980
co2_scenario         = "SSP2-4.5"  # SSP1-2.6, SSP2-4.5, or SSP5-8.5
exposure_conditions  = 2
ad                   = 0.0      # Pozzolanic material in the concrete, relative to cement mass (%) — Possan et al. (2016) model input.
                                 # cement_type=3 is CP II F (limestone filler, not pozzolanic), and there is no record of a
                                 # supplementary pozzolanic addition to this 1980 mix — ad=10 was inherited from matching the
                                 # old ML surrogate's training assumption, not from anything specific to this structure. ad
                                 # raises predicted carbonation depth by ~12% at any age (it depletes the alkaline reserve
                                 # faster), so ad=0 is the less conservative (slower-carbonation) of the two.
n_samples            = 2000      # Number of design samples
n_latent_samples     = 2500     # Number of latent samples per design sample. Also the filename prefix
n_samples_validation = 500       # Number of validation samples, redrawn at every time step
n_lambdas            = 4        # Number of λs (λ1, λ2, λ3, λ4)
latent_cov           = 0.15    # CoV of the RH/fck/cover latent multipliers. 2% is low for real execution
                                 # variability (15-30%+ is routinely reported for cover/strength); fck and cover
                                 # are drawn Lognormal (can't go negative), RH is drawn Normal truncated to [0, 100]

## **3. Design samples**

In [10]:
fck_dist = Uniform(loc=fck_min, scale=fck_max - fck_min)
rh_dist  = Uniform(loc=rh_min, scale=rh_max - rh_min)
cov_dist = Uniform(loc=cov_min, scale=cov_max - cov_min)
joint    = JointIndependent(marginals=[fck_dist, rh_dist, cov_dist])

x_pce_rvs = joint.rvs(n_samples)
x_val = joint.rvs(n_samples_validation)

print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations per time step: {(n_samples + n_samples_validation) * n_latent_samples}")
print("\nSample statistics:")
print(f"   fck:   {x_pce_rvs[:, 0].min():.1f} - {x_pce_rvs[:, 0].max():.1f} MPa (mean: {x_pce_rvs[:, 0].mean():.1f} MPa)")
print(f"   RH:    {x_pce_rvs[:, 1].min():.1f} - {x_pce_rvs[:, 1].max():.1f}% (mean: {x_pce_rvs[:, 1].mean():.1f}%)")
print(f"   cover: {x_pce_rvs[:, 2].min():.1f} - {x_pce_rvs[:, 2].max():.1f} mm (mean: {x_pce_rvs[:, 2].mean():.1f} mm)")

Samples generated successfully!
   Number of design samples: 2000
   Number of latent samples per design sample: 2500
   Total simulations per time step: 6250000

Sample statistics:
   fck:   20.0 - 50.0 MPa (mean: 34.7 MPa)
   RH:    20.0 - 79.9% (mean: 50.3%)
   cover: 15.0 - 60.0 mm (mean: 37.1 mm)


## **4. Time grid**

In [11]:
times = np.linspace(0, 100, 5, endpoint=True)  # Time points for carbonation depth prediction
times

array([  0.,  25.,  50.,  75., 100.])

## **5. Generate the dataset at each time step**

The same `x_val` (drawn once, in section 3) is reused across every time step — so the same
validation design points can be tracked over time, instead of being redrawn independently at each
`t`. `generate_dataset_at_time_durability` does the rest: latent sampling, carbonation depth
prediction, GLD fit, and saving `dataset_full`/`dataset_unique` for both splits.

In [12]:
print("="*60)
print("GENERATING THE DURABILITY DATASET")
print("="*60)

generation_results = []
for t in times:
    result = generate_dataset_at_time_durability(
                                                    x_train=x_pce_rvs,
                                                    x_val=x_val,
                                                    time_step=t,
                                                    cement_type=cement_type,
                                                    installation_year=installation_year,
                                                    co2_scenario=co2_scenario,
                                                    exposure_conditions=exposure_conditions,
                                                    ad=ad,
                                                    n_latent_samples=n_latent_samples,
                                                    latent_cov=latent_cov,
                                                    output_dir='.',
                                                )
    generation_results.append(result)

GENERATING THE DURABILITY DATASET

----------------------------------------
GENERATING DATASET FOR TIME STEP: 0.0 years
----------------------------------------


KeyboardInterrupt: 

## 6. Timing summary

Cost of building the dataset, per time step.

In [ ]:
timing_rows = []
for result in generation_results:
    train_t = result['df_unique_train']['Processing time (s)']
    val_t   = result['df_unique_val']['Processing time (s)']
    timing_rows.append({
                           'Time (years)':   result['time_step'],
                           'n_train':        len(train_t),
                           'Train total (s)': train_t.sum(),
                           'Train mean (ms)': train_t.mean() * 1e3,
                           'n_val':          len(val_t),
                           'Val total (s)':  val_t.sum(),
                       })

emulator_timing = pd.DataFrame(timing_rows)

with open(f'{n_latent_samples}_emulator_timing_durability.pkl', 'wb') as f:
    dill.dump(emulator_timing, f)

train_total = emulator_timing['Train total (s)'].sum()
val_total   = emulator_timing['Val total (s)'].sum()

print(f"Train split - emulator g-value dataset generation time: {train_total:.1f} s")
print(f"Val split   - emulator g-value dataset generation time: {val_total:.1f} s")
print(f"Total emulator g-value dataset generation time (train + val): {train_total + val_total:.1f} s")
emulator_timing

Train split - emulator g-value dataset generation time: 1803.1 s
Val split   - emulator g-value dataset generation time: 453.7 s
Total emulator g-value dataset generation time (train + val): 2256.8 s


,Time (years),n_train,Train total (s),Train mean (ms),n_val,Val total (s)
0,0.0,2000,362.077174,181.038587,500,91.862942
1,25.0,2000,366.391805,183.195902,500,91.729051
2,50.0,2000,358.595040,179.297520,500,89.724485
3,75.0,2000,357.628061,178.814031,500,89.884486
4,100.0,2000,358.440037,179.220018,500,90.461438


## 7. Unique dataset statistics

Concatenate the `dataset_unique` (train + val, all time steps) frames already held in
`generation_results` and describe the columns, including the fitted lambdas.

In [ ]:
dataset_unique_all = pd.concat(
    [
        result[f'df_unique_{split}'].assign(split=split)
        for result in generation_results
        for split in ('train', 'val')
    ],
    ignore_index=True,
)

print(f"Combined unique dataset: {len(dataset_unique_all)} rows "
      f"({len(generation_results)} time steps x train/val splits)")
dataset_unique_all.describe()

Combined unique dataset: 12500 rows (5 time steps x train/val splits)


,fck,rh,cov,lambda 1,lambda 2,lambda 3,lambda 4,Processing time (s)
count,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000
mean,34.976486,50.233637,37.190765,25.151379,0.244588,0.198312,0.130039,0.180544
std,8.741789,17.516755,13.089529,15.991166,0.099337,0.578944,0.526185,0.006433
min,20.008795,20.019702,15.031070,-33.442005,0.004084,-0.066495,-0.101100,0.167420
25%,27.357709,35.016954,25.905700,13.816824,0.175714,0.092993,0.058530,0.176329
50%,35.168931,50.282784,37.420649,25.234626,0.216083,0.159178,0.078915,0.179558
75%,42.488472,65.514818,48.393525,37.580551,0.289068,0.217479,0.109110,0.183584
max,49.999191,79.994533,59.954782,60.605496,0.698412,9.031806,8.875910,0.292797
